# 02 — Agent Evaluation

**PPE Compliance Agent — ITAI 1378 Final Project**

Evaluates the agent at two levels, per course requirements:

1. **Component level** — the CV model's own metrics (mAP, precision, recall) from training
2. **System level** — task success rate across test scenarios, robustness to bad inputs, efficiency, and honest failure analysis

**Run the Setup cell first, every time**, especially after a fresh Colab session or a Runtime restart — it clones the repo, installs dependencies, and puts you in the right folder. Skipping it or running cells out of order is the #1 cause of import errors here.

## 0. Setup — run this first, every session

Edit `REPO_URL` below to your own GitHub repo URL once, then just run this cell.
If anything below ever gets confused (stale imports, path errors), use **Runtime → Restart session**, then run this cell again from a clean start — don't just re-run cells on top of a broken state.

In [ ]:
import os

REPO_URL = "https://github.com/Huynguyen-175/ITAI1378_Final_PPEComplianceAgent.git"
REPO_DIR = "/content/ITAI1378_Final_PPEComplianceAgent"

# Clean up any previous/broken clone so we always start from a known state
!rm -rf {REPO_DIR}
!git clone -q {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print("Repo root:", os.getcwd())

!pip install -q -r requirements.txt

os.chdir(f"{REPO_DIR}/notebooks")
print("Now in:", os.getcwd())

# Self-check: confirm the install actually worked before moving on.
# If this fails, everything below will fail too — fix it here first.
try:
    import ultralytics
    print(f"\nultralytics OK — version {ultralytics.__version__}")
except ImportError as e:
    print(f"\nSETUP FAILED: {e}")
    print("Try running this Setup cell again. If it still fails, run:")
    print('  !pip install -q ultralytics')
    print("as a one-off, then re-run this cell.")

print("\nIMPORTANT: click Runtime -> Restart session now, then re-run this Setup cell once more before continuing.")


## 0b. (Optional) Load real trained weights from Google Drive

Skip this cell if you haven't trained yet — the agent will automatically fall back to base `yolov8n.pt` and still run end-to-end, just without real mask-detection classes.

This version **searches your whole Drive** for `best_yolov8n_ppe.pt` instead of assuming a fixed folder name, so it won't break if your save location doesn't match what a notebook expects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import glob, shutil, os

matches = glob.glob("/content/drive/MyDrive/**/best_yolov8n_ppe.pt", recursive=True)

if not matches:
    print("No file named 'best_yolov8n_ppe.pt' found anywhere in your Drive.")
    print("If you haven't trained yet, that's fine — skip this cell, the agent will use fallback weights.")
    print("If you HAVE trained, double check the exact filename with:")
    print('  !find "/content/drive/MyDrive" -iname "*.pt"')
else:
    src = matches[0]
    if len(matches) > 1:
        print(f"Found {len(matches)} matches, using the first one:")
        for m in matches:
            print(" ", m)
    os.makedirs("../models/trained", exist_ok=True)
    shutil.copy(src, "../models/trained/best_yolov8n_ppe.pt")
    print(f"\nCopied from: {src}")
    print("       to:  ../models/trained/best_yolov8n_ppe.pt")

## 1. Import the agent

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # repo root, so `agents` and `tools` import cleanly

from agents.ppe_compliance_agent import PPEComplianceAgent

## 2. Component-Level Metrics (from training)

In [ ]:
component_metrics = {
    "mAP@0.5": 0.900,
    "mAP@0.5:0.95": 0.608,
    "precision": 0.934,
    "recall": 0.825,
    "epochs_trained": 75,
    "training_time_min": 76.4,
}
for k, v in component_metrics.items():
    print(f"{k:20s}: {v}")

## 3. System-Level Evaluation — Task Success Rate

Runs the full agent (all 6 pipeline stages) on a labeled test set and compares the
agent's final decision against ground truth. This measures the *agent's* accuracy,
not just the detector's.

**To run this for real:** point `TEST_DIR` at the Roboflow `test/images` folder
(download it below, or reuse the one from `01_exploration.ipynb` if you still have
that Colab session open) and build `GROUND_TRUTH` from the matching YOLO label files
(`with_mask` → COMPLIANT, `without_mask`/`incorrectly_worn_mask` → NON_COMPLIANT).
Aim for at least 10-20 scenarios per the assignment requirement — the 3 bundled
sample images below are only a smoke test, not a full evaluation.

In [ ]:
# Optional: download the full test set here if you want to evaluate on it directly
# instead of just the 3 bundled sample images.
#
# import roboflow
# roboflow.login()
# rf = roboflow.Roboflow()
# project = rf.workspace("agh-ett2f").project("mask-detection-yolov8")
# dataset = project.version(16).download("yolov8")
# TEST_DIR = f"{dataset.location}/test/images"

TEST_DIR = "../data/sample"  # swap for the full test/images folder above for a real 10-20 scenario run

# Ground truth for the 3 bundled sample images (manually confirmed).
# If you switch TEST_DIR to the full Roboflow test set, build this dict from the
# YOLO label .txt files instead (each line's class id tells you the ground truth).
GROUND_TRUTH = {
    "caregiver_car_nomask.jpg": "NON_COMPLIANT",
    "caregiver_home_nomask.jpg": "NON_COMPLIANT",
    "caregiver_clinic_masked.jpg": "COMPLIANT",
}

agent = PPEComplianceAgent(
    weights_path="../models/trained/best_yolov8n_ppe.pt",  # falls back gracefully if missing
    results_dir="../results",
)
traces = agent.run(TEST_DIR)

In [ ]:
import os

correct = 0
total = 0
rows = []
for t in traces:
    fname = os.path.basename(t["image_path"])
    truth = GROUND_TRUTH.get(fname)
    if truth is None:
        continue
    predicted = t["status"]
    is_correct = predicted == truth
    correct += int(is_correct)
    total += 1
    rows.append((fname, truth, predicted, is_correct))

print(f"{'Image':35s} {'Ground Truth':15s} {'Predicted':15s} {'Correct'}")
for r in rows:
    print(f"{r[0]:35s} {r[1]:15s} {r[2]:15s} {r[3]}")

print(f"\nTask success rate: {correct}/{total} = {correct/max(total,1)*100:.1f}%")
print("NOTE: if this ran on the 3 bundled sample images only, that's a smoke test.")
print("Re-run with TEST_DIR pointed at the full Roboflow test/ split for the required 10-20 scenario evaluation.")

## 4. Robustness — Bad Input Handling

In [ ]:
import os
from PIL import Image

os.makedirs("../data/robustness_test", exist_ok=True)

# corrupt file (garbage bytes with .jpg extension)
with open("../data/robustness_test/corrupt.jpg", "w") as f:
    f.write("this is not an image")

# tiny image, below usable size
Image.new("RGB", (10, 10), "white").save("../data/robustness_test/tiny.jpg")

# valid but blank image (no useful content)
Image.new("RGB", (640, 640), "white").save("../data/robustness_test/blank.jpg")

robustness_traces = agent.run("../data/robustness_test")

for t in robustness_traces:
    print(f"{os.path.basename(t['image_path']):20s} -> {t['status']:25s} ({t['preprocessing']['reason']})")

**Result:** the corrupt file and the undersized image are both caught at the preprocessing
stage and marked `SKIPPED_INVALID_INPUT` — the agent logs why and moves on to the next
image rather than crashing. The blank (but valid) image is processed normally and correctly
returns `NO_DETECTION`, since there's genuinely nothing to detect — the agent abstains
instead of guessing.

## 5. Efficiency

Average per-image latency is also written to `results/metrics.txt` after every run
(`agent._write_batch_summary()`). Check that file for the running average across all
batches, or compute it directly from the traces below.

In [ ]:
latencies = [t["latency_sec"] for t in traces if "latency_sec" in t]
if latencies:
    print(f"Average latency: {sum(latencies)/len(latencies):.3f} sec/image")
    print(f"Min: {min(latencies):.3f}s | Max: {max(latencies):.3f}s")

## 6. Honest Failure Analysis

**Required: at least 2 documented failure cases with explanation.**

### Failure Case 1 (confirmed, real): false positive overrides correct detections

**Image:** `data/sample/caregiver_clinic_masked.jpg` — two people, both actually wearing masks.

**What happened:** the agent correctly detected both real masks at high confidence (`with_mask` 0.90 and 0.87), but also produced a spurious third detection (`without_mask`, confidence only 0.55) on a small orange object clipped to one person's scrub top — not a face at all, likely an ID badge holder or pen case. Because Rule R1 treats *any* `without_mask` detection as an automatic override regardless of confidence, this single low-confidence false positive flipped the entire frame's verdict from what should have been COMPLIANT to NON_COMPLIANT — even though both genuine detections were correct and far more confident than the false one.

**Raw detections from the trace** (`results/traces/caregiver_clinic_masked_*.json`):
```json
{"class_name": "with_mask", "confidence": 0.9033, "bbox_xyxy": [552.5, 146.2, 661.5, 281.1]},
{"class_name": "with_mask", "confidence": 0.8696, "bbox_xyxy": [262.7, 114.5, 355.5, 221.3]},
{"class_name": "without_mask", "confidence": 0.5475, "bbox_xyxy": [146.4, 252.8, 186.4, 306.4]}
```
Note the false positive's box is roughly 40×54px — much smaller than the two real face detections — and sits in the lower-left of the frame, nowhere near either person's face.

**Root cause:** a design weakness in the reasoning layer (`agents/reasoning.py`), not a training/data problem — Rule R1 gives equal veto power to a 0.55-confidence detection and a 0.90-confidence detection.

**Fix applied and verified** (`agents/reasoning.py`, v2): violation classes now require confidence ≥ 0.65 (`VIOLATION_CONFIDENCE_FLOOR`) before they're allowed to override a `with_mask` detection in the same frame — a higher bar than the base 0.4 detection threshold. Low-confidence violation candidates are still logged (`suppressed_detections` in every trace) for audit transparency; they just no longer unilaterally flip the result. Re-running the exact detections from this failure case through the fixed logic now correctly returns **COMPLIANT**, with the suppressed 0.55-confidence detection noted in the explanation rather than silently ignored or allowed to override. Regression-tested against the two genuine high-confidence violation cases from `results/demo_non_compliant_*_realmodel.png` (0.87, 0.90) to confirm real violations still correctly trigger NON_COMPLIANT.

### Failure Case 2 — TODO

*(Run Section 3 against the full Roboflow `test/` split — not just the 3 bundled samples — and document a second real failure case here. Known/anticipated failure modes worth checking for: occluded or side-profile faces producing `NO_DETECTION`; domain mismatch between the training data and “in-the-wild” photo conditions producing lower/unstable confidence scores.)*